In [ ]:
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from mask_dataset import WheatBinaryDataset

In [ ]:
DEVICE = "cuda"

# Training Config
train_transform = A.Compose([
    A.Normalize(),
    ToTensorV2(),
])


In [ ]:
dataset = WheatBinaryDataset('masks/train', transform=train_transform)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)


In [ ]:
model = smp.DeepLabV3Plus(
    encoder_name="mobilenet_v2",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1, # Binary segmentation
).to(DEVICE)


In [ ]:
criterion = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
model.train()
for epoch in range(100):
    epoch_loss = 0
    print(f"\n--- Starting Epoch {epoch+1} ---", flush=True)

    for batch_idx, (images, masks) in enumerate(train_loader):
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # Print every 5 batches to show signs of life
        if batch_idx % 5 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}", flush=True)

    avg_loss = epoch_loss / len(train_loader)
    print(f"--- Epoch {epoch+1} Finished | Avg Loss: {avg_loss:.4f} ---", flush=True)

In [7]:
@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    metric_iou = 0
    metric_f1 = 0

    # We use a threshold of 0.5 to turn the model's Sigmoid output into a binary 0/1 mask
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)

        # Forward pass
        output = model(images)

        # Apply sigmoid and threshold
        tp, fp, fn, tn = smp.metrics.get_stats(
            output,
            masks.long(),
            mode='binary',
            threshold=0.5
        )

        # Calculate IoU and F1 (Dice) for this batch
        metric_iou += smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")
        metric_f1 += smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro")

    avg_iou = metric_iou / len(loader)
    avg_f1 = metric_f1 / len(loader)

    return avg_iou, avg_f1

In [8]:
val_dataset = WheatBinaryDataset('masks/valid', transform=train_transform) # or your val_transform
val_loader = DataLoader(val_dataset, batch_size=16, num_workers=4)

test_dataset = WheatBinaryDataset('masks/test', transform=train_transform)
test_loader = DataLoader(test_dataset, batch_size=16, num_workers=4)

# Evaluation
val_iou, val_f1 = evaluate_model(model, val_loader, DEVICE)
test_iou, test_f1 = evaluate_model(model, test_loader, DEVICE)

print(f"Validation mIoU: {val_iou:.4f} | Validation F1 (Dice): {val_f1:.4f}")
print(f"Test mIoU: {test_iou:.4f} | Test F1 (Dice): {test_f1:.4f}")

Pre-loading images into RAM for speed...
Pre-loading images into RAM for speed...
Validation mIoU: 0.9548 | Validation F1 (Dice): 0.9768
Test mIoU: 0.9156 | Test F1 (Dice): 0.9551


In [9]:
torch.save(model.state_dict(), 'best_canopy_seg_model.pth')

print("Model weights saved successfully.")

Model weights saved successfully.
